# PDB preparation for alpha-galactosidase A

This notebook prepares apo and holo systems of α-galactosidase A for MD simulations (untill equilibration).

Remember to activate the conda environment with ```conda activate ace_software```.
If the required software is not installed yet, please read the README file in this repository.




### Sections
First, run the following cells to initiate the correct apo and DGJ mol objects.

Then:  
[APO only build](#apo)  
[DGJ only build](#holo-(DGJ))  

In [1]:
#IMPORT PACKAGES
from htmd.ui import *
import pandas as pd 
import numpy as np
import os 
from htmd.builder import amber
from acemd.protocols import setup_equilibration
import re 


2026-01-30 16:21:20,257 - numexpr.utils - INFO - Note: detected 128 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
2026-01-30 16:21:20,268 - numexpr.utils - INFO - Note: NumExpr detected 128 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2026-01-30 16:21:20,269 - numexpr.utils - INFO - NumExpr defaulting to 16 threads.



Please cite HTMD: Doerr et al.(2016)JCTC,12,1845. https://dx.doi.org/10.1021/acs.jctc.6b00049
HTMD Documentation at: https://software.acellera.com/htmd/

You are on the latest HTMD version (2.5.7).



2026-01-30 16:21:24,754 - acemd - INFO - # You are on the latest ACEMD version (4.0.17).



Acellera Software Not-for-Profit License Agreement v1.2

The software ("Software") has been developed by the contributing
researchers from Acellera and made available through Acellera for your
internal, non-profit research use.

Acellera allows researchers at your institution to run, display, copy and
modify Software on the following conditions:

1. The Software remains at your institution and is not published, distributed,
   or otherwise transferred or made available to other than institution employees
   and students involved in research under your supervision.

2. You agree to make results generated using Software available to other
   academic researchers for non-profit research purposes. If you wish to obtain
   Software for any commercial purposes, including fee-based service projects, you
   will need to execute a separate licensing agreement with Acellera and pay a
   fee. In that case please contact: info@acellera.com.

3. You retain in Software and any modifications to Soft

The following cell contains:
- PART 1: modifiable data;
- PART 2: fixed patches for the glycans, glycosylation sites and ligand import files;
- PART 3: Molecule preparation and cleanup.


In [2]:
###################### PART 1 ######################
###################### CHANGE DATA AS PLEASED 

parent_folder='prepared_systems' #main folder different PDB-generated systems are stored
molecule_folder = '3s5y_amber' #parent folder for organising wt/mut and apo/holo
folder = f'{parent_folder}/{molecule_folder}' #final case subfolder

molecule = '../glycosylation/3s5y_amber_fixed.pdb' #PDB path
#the PDB is obtained from rcsb.org and it is expected to be modelled with glycoshape.org

#IF NOTHING TO REMOVE, LEAVE THE LISTS EMPTY.
mutations = [] #resid to mutate. #('resid 215', 'SER'),('resid 301', 'GLN')
#if no mutations, wt systems will be prepared

resnames_to_remove = ['4YB', 'VMB', '0MA']  #'SO4','HOH', 'NOJ'
#NOTE if the molecule is previously processed by glycoshape, water and ions should be already removed

# EQUILIBRATION AND SIMULATION DATA
eq_run = '50 ns' #also in us, ns, ps and fs
eq_temp = 300
minimize = 1000
prod_run = '1 us' #also in us, ns, ps and fs
prod_temp = 300



###################### PART 2 ######################
###################### DO NOT CHANGE FROM HERE


#glycosylation site
gly_resid = [139, 192, 215]
#the three glycosilation sites have been evaluated in literature

#correct ligand for Holo str
#to model alternative ligands, replace these files with the corresponding
#CGenFF-parameterized structures for chain A and chain B, keeping the rest
#of the protocol unchanged.

dgj_a = '../DGJ/3s5y/DGJ_A_cgenff.mol2' #chain A 
dgj_b = '../DGJ/3s5y/DGJ_B_cgenff.mol2' #chain B

#MAKE FOLDER FOR {MOLECULE}
os.makedirs(f'../{folder}', exist_ok=True) #check and make

#AMINO ACID CODES FOR CONVERSION (do not change)
aa_3to1 = {'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F', 'GLY': 'G', 
 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L', 'MET': 'M', 'ASN': 'N', 
 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R', 'SER': 'S', 'THR': 'T', 'VAL': 'V', 
 'TRP': 'W', 'TYR': 'Y'} #conversion from 3-letter code to 1-letter code

print(f'Protein structure: {molecule_folder}, mutated residues: {mutations}, glycosilated residues: {gly_resid}')


###################### PART 3 ######################
###################### COMMON MOLECULE PREPARATION PART 

#read the molecule pdb, remove the unwanted molecules 
mol = Molecule(molecule)

#REMOVE RESNAMES
print(f'\nRemoving residues: {resnames_to_remove}') #check
original_count = mol.numAtoms
if resnames_to_remove:
    selection = ' or '.join([f'resname {res}' for res in resnames_to_remove]) #build correct selection
    mol.filter(f'not ({selection})') #remove
removed_count = original_count - mol.numAtoms #check
print(f'Removed {removed_count} atoms')
print(f'Remaining atoms: {mol.numAtoms}')

#CORRECT NUMBERING OF THE RESIDUES IN THE PROTEIN CHAINS


#CREATE DUPLICATE FOR MOL_APO 
mol_apo = mol.copy()
mol_DGJ = mol.copy()

print('Systems are ready to be prepared.')

Protein structure: 3s5y_amber, mutated residues: [], glycosilated residues: [139, 192, 215]

Removing residues: ['4YB', 'VMB', '0MA']


2026-01-30 16:21:30,208 - moleculekit.molecule - INFO - Removed 708 atoms. 6255 atoms remaining in the molecule.


Removed 708 atoms
Remaining atoms: 6255
Systems are ready to be prepared.


From this point on apo and holo are run from different cells:

[APO only build](#apo)  
[DGJ only build](#holo-(DGJ))  

## APO (da sistemare)
[Back to main](#sections) 

In [ ]:
#HANDLE MUTATIONS
proteins_apo=[]

if mutations: #<resid  resid> and <mut resname>
    for mutation in mutations:
        resid_sel, new_res = mutation 
        wt = np.unique(mol_apo.get('resname', resid_sel))
        wt = aa_3to1[wt[0]] 
        num = resid_sel.split(' ')[1]
        mut = aa_3to1[new_res]

        folder_apo = f'../{folder}/apo_{wt}{num}{mut}'
        os.makedirs(folder_apo, exist_ok=True)
        print(f'Storing generated data at {folder_apo}.')
        
        #MAKE A COPY OF THE APO TO BE MUTATED
        print(f'Mutating protein at {mutation}')
        mol_apo_mut = mol_apo.copy()
        mol_apo_mut.mutateResidue(resid_sel, new_res)
            
        proteins_apo.append((f'apo_{wt}{num}{mut}', mol_apo_mut, folder_apo, num)) 

#FOR WILD TYPE
else: #no mutations
    print('No mutation specified: preparing a wild-type system.')
    folder_apo = f'../{folder}/apo'
    os.makedirs(folder_apo, exist_ok=True) 
    proteins_apo.append(('apo', mol_apo, folder_apo, None)) 



#COMMON
for label, mol, folder_apo, num in proteins_apo:
    segid_to_remove = [] #in case of mut 
    print(f"\nProcessing system: {label}")

    #SYSTEM PREPARATION AND SEGMENTATION.
    system_apo, data = systemPrepare(mol, pH=7.0, return_details=True, plot_pka=f'{folder_apo}/{molecule_folder}_{label}_pka')
    system_apo = autoSegment(system_apo) #segment system
    
#GUARDARE SE C'è GLICANO IN SITO DI MUTAZIONE


    #SYSTEM SOLVATION
    system_solv_apo = solvate(system_apo, negx = 20  , negy = 20, negz = 20, posx = 20, posy = 20, posz = 20)
    system_solv_apo.write(f'{folder_apo}/{molecule_folder}_{label}_solv.pdb')
    #system_solv.write(f'{folder_apo}/{molecule}_solv.psf') #non giusto

    #SYSTEM BUILDING WITH CHARMM36m AND PATCHES
    #other parameters available
    system_amber_apo = amber.build(system_solv_apo,  saltconc = 0.15, saltanion = 'CL', saltcation = 'K',
                                ff= ['leaprc.protein.ff14SB', 'leaprc.GLYCAM_06j-1', 'leaprc.water.tip3p', 'leaprc.gaff2'], 
                                outdir = f'{folder_apo}/build') #patches = patch,
    print('build/ folder generated.') 

    #MINIMIZATION AND EQUILIBRATION PREPARATION
    setup_equilibration(builddir=f'{folder_apo}/build', 
                        outdir=f'{folder_apo}/equilibration',
                        run = eq_run, #also in us, ns, ps and fs
                        temperature = eq_temp,
                        coordinates = f'{folder_apo}/build/structure.pdb',
                        structure = f'{folder_apo}/build/structure.psf',
                        parameters = f'{folder_apo}/build/parameters.prm',
                        minimize = minimize)
    print('equilibration/ folder generated.') 

#to remember:
# "NA","MG","ZN","K","CS","CA","CL"  
#, 'noj/noj_g.rtf'
#charmm.listFiles()  #check for files  
# 


#### The **production** folder can be generated only **after the equilibration is compleded**.

In particular:

1. run equilibration
2. check equilibration ended with *check_end.py* 
3. run *production_prep.py*
4. run production

## HOLO (DGJ)
[Back to main](#sections) 

In [ ]:
#HANDLE MUTATIONS
proteins_DGJ=[]

if mutations: #<resid  resid> and <mut resname>
    for mutation in mutations:
        resid_sel, new_res = mutation 
        wt = np.unique(mol_DGJ.get('resname', resid_sel))
        wt = aa_3to1[wt[0]] 
        num = resid_sel.split(' ')[1]
        mut = aa_3to1[new_res]

        folder_DGJ = f'../{folder}/DGJ_{wt}{num}{mut}'
        os.makedirs(folder_DGJ, exist_ok=True)
        print(f'Storing generated data at {folder_DGJ}.')
        
        #MAKE A COPY OF THE APO TO BE MUTATED
        print(f'Mutating protein at {mutation}')
        mol_DGJ_mut = mol_DGJ.copy()
        mol_DGJ_mut.mutateResidue(resid_sel, new_res)
           
        proteins_DGJ.append((f'DGJ_{wt}{num}{mut}', mol_DGJ_mut, folder_DGJ, num))  

#FOR WILD TYPE
else: #no mutations
    print('No mutation specified: preparing a wild-type system.')
    folder_DGJ = f'../{folder}/DGJ'
    os.makedirs(folder_DGJ, exist_ok=True) 
    proteins_DGJ.append(('DGJ', mol_DGJ, folder_DGJ, None))  



#COMMON
for label, mol, folder_DGJ, num in proteins_DGJ: 
    segid_to_remove = [] # in case of mut
    print(f"\nProcessing system: {label}")

    #APPEND CORRECT DGJ IN CHAIN A AND B
    #chain A
    DGJ_A = Molecule(dgj_a)
    DGJ_A.set('resname', 'DGJ')
    DGJ_A.set('resid', '1', 'resname DGJ')
    DGJ_A.set('chain', 'L', 'resname DGJ')
    DGJ_A.set('segid', 'P8', 'resname DGJ')
    
    #chain B
    DGJ_B = Molecule(dgj_b)
    DGJ_B.set('resname', 'DGJ')
    DGJ_B.set('resid', '1', 'resname DGJ')
    DGJ_B.set('chain', 'M', 'resname DGJ')
    DGJ_B.set('segid', 'P9', 'resname DGJ')
    #append
    #mol.append(DGJ_A)
    #mol.append(DGJ_B)

    #SYSTEM PREPARATION AND SEGMENTATION.
    system_DGJ, data = systemPrepare(mol, pH=7.0, return_details=True, plot_pka=f'{folder_DGJ}/{molecule_folder}_{label}_pka', force_protonation=[("chain A and resid 203", "GLU"),("chain B and resid 203", "GLU")], ignore_ns_errors=True)
    system_DGJ = autoSegment(system_DGJ, basename='P') #segment system

    print('Appending one ligand (DGJ) in each monomer.')
    system_DGJ.append(DGJ_A)
    system_DGJ.append(DGJ_B)

    #intermediate saving
    data.to_csv(f'{folder_DGJ}/{molecule_folder}_{label}_prep.csv') #save
    system_DGJ.write(f'{folder_DGJ}/{molecule_folder}_{label}_prep.pdb') #save

    segments_DGJ = np.unique(system_DGJ.segid) #check
    print(f'Final segments: {segments_DGJ}')

#GUARDARE SE C'è GLICANO IN SITO DI MUTAZIONE

    #SYSTEM SOLVATION
    system_solv_DGJ = solvate(system_DGJ, negx = 20  , negy = 20, negz = 20, posx = 20, posy = 20, posz = 20)
    system_solv_DGJ.write(f'{folder_DGJ}/{molecule_folder}_{label}_solv.pdb')
    #system_solv.write(f'../{folder}/_solv.psf') #non giusto

    #SYSTEM BUILDING WITH AMBER
    #other parameters available
    system_amber_DGJ = amber.build(system_solv_DGJ,  saltconc = 0.15, saltanion = 'CL', saltcation = 'K',
                                ff= ['leaprc.protein.ff14SB', 'leaprc.GLYCAM_06j-1', 'leaprc.water.tip3p', 'leaprc.gaff2'],    
                                topo=['../DGJ/3s5y/dgj.prepi'],
                                param=['../DGJ/3s5y/dgj.frcmod'],
                                  outdir = f'{folder_DGJ}/build') 
    print('build/ folder generated.')                         

    #MINIMIZATION AND EQUILIBRATION PREPARATION
    setup_equilibration(builddir=f'{folder_DGJ}/build', 
                        outdir=f'{folder_DGJ}/equilibration',
                        run = eq_run, 
                        temperature = eq_temp,
                        coordinates = f'{folder_DGJ}/build/structure.crd',
                        structure = f'{folder_DGJ}/build/structure.pdb',
                        parameters = f'{folder_DGJ}/build/parameters.prmtop',
                        minimize = minimize)
    print('equilibration/ folder generated.')


    #to remember:
    # "NA","MG","ZN","K","CS","CA","CL'  
    #, 'noj/noj_g.rtf'
    #charmm.listFiles()  #check for files

No mutation specified: preparing a wild-type system.

Processing system: DGJ

---- Molecule chain report ----
Chain A:
    First residue: LEU    32  
    Final residue: MET   421  
Chain B:
    First residue: LEU    32  
    Final residue: GLN   422  
---- End of chain report ----



2026-01-30 16:21:31,868 - moleculekit.tools.preparation - INFO - Forcing protonation of residue A:203 to GLU
2026-01-30 16:21:31,920 - moleculekit.tools.preparation - INFO - Forcing protonation of residue B:203 to GLU
2026-01-30 16:21:36,422 - moleculekit.tools.preparation - INFO - Skipping titration of residue GLU:A:203
2026-01-30 16:21:36,443 - moleculekit.tools.preparation - INFO - Skipping titration of residue GLU:B:203
2026-01-30 16:21:41,885 - moleculekit.tools.preparation - WARNING - The following residues have not been optimized: NLN
2026-01-30 16:21:43,997 - moleculekit.tools.preparation - INFO - Modified residue GLU    48 A to GLH
2026-01-30 16:21:44,227 - moleculekit.tools.preparation - INFO - Modified residue ASP   136 A to ASH
2026-01-30 16:21:44,423 - moleculekit.tools.preparation - INFO - Modified residue ASP   234 A to ASH
2026-01-30 16:21:44,446 - moleculekit.tools.preparation - INFO - Modified residue GLU    48 B to GLH
2026-01-30 16:21:44,447 - moleculekit.tools.prep

Appending one ligand (DGJ) in each monomer.


2026-01-30 16:22:03,723 - htmd.builder.solvate - INFO - Using water pdb file at: /leonardo/home/userexternal/icazzani/miniconda3/envs/ace_software/lib/python3.10/site-packages/htmd/share/solvate/wat.pdb


Final segments: ['P0' 'P1' 'P8' 'P9']


...

Disulfide Bond between: UniqueResidueID<resname: 'CYX', chain: 'B', resid: 418, insertion: '', segid: 'P1'>
                   and: UniqueResidueID<resname: 'CYX', chain: 'B', resid: 425, insertion: '', segid: 'P1'>

Disulfide Bond between: UniqueResidueID<resname: 'CYX', chain: 'A', resid: 26, insertion: '', segid: 'P0'>
                   and: UniqueResidueID<resname: 'CYX', chain: 'A', resid: 33, insertion: '', segid: 'P0'>

Disulfide Bond between: UniqueResidueID<resname: 'CYX', chain: 'A', resid: 22, insertion: '', segid: 'P0'>
                   and: UniqueResidueID<resname: 'CYX', chain: 'A', resid: 64, insertion: '', segid: 'P0'>

Disulfide Bond between: UniqueResidueID<resname: 'CYX', chain: 'A', resid: 348, insertion: '', segid: 'P0'>
                   and: UniqueResidueID<resname: 'CYX', chain: 'A', resid: 352, insertion: '', segid: 'P0'>

Disulfide Bond between: UniqueResidueID<resname: 'CYX', chain: 'B', resid: 414, insertion: '', segid: 'P1'>
                   and: Uniq

2026-01-30 16:22:41,367 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2026-01-30 16:22:44,057 - htmd.builder.amber - INFO - Starting the build.
2026-01-30 16:23:16,090 - htmd.builder.amber - INFO - Finished building.
2026-01-30 16:23:19,893 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2026-01-30 16:23:20,276 - moleculekit.writers - WARNING - Field "resid" of PDB overflows. Your data will be truncated to 4 characters.
2026-01-30 16:24:27,778 - htmd.builder.builder - WARNING - Found cis peptide bond in 1 frames: [0] in the omega diheral "Angle of (ASN 349 CA  ) (ASN 349 C  ) (PRO 350 N  ) (PRO 350 CA  ) " with indexes [5403, 5413, 5415, 5425]
2026-01-30 16:24:27,901 - htmd.builder.builder - WARNING - Found cis peptide bond in 1 frames: [0] in the omega diheral "Angle of (LEU 358 CA  ) (LEU 358 C  ) (PRO 359 N  ) (PRO 359 CA  ) " with indexes [5540, 5555, 55

build/ folder generated.


2026-01-30 16:29:08,945 - moleculekit.readers - WARNING - Non-integer values were read from the PDB "serial" field. Dropping PDB values and assigning new ones.
2026-01-30 16:29:09,032 - moleculekit.readers - WARNING - Reading PDB file with more than 99999 atoms. Bond information can be wrong.


equilibration/ folder generated.


#### The **production** folder can be generated only **after the equilibration is compleded**.

In particular:

1. run equilibration
2. check equilibration ended with *check_end.py* 
3. run *production_prep.py*
4. run production
 